In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np

from libs.dataset import (
    anonymize_dataset,
    check_no_collision,
    create_dataset,
    download_images,
    generate_increasing_datasets,
    join_dataframe,
    split_dataset,
)
from libs.plotting import *
from libs.train import *
from libs.utils import link_dir, logscale

IF YOU NEVER DOWNLOAD LOCALY IMAGES + ANONYMISE

In [ ]:
root = find_root_repository()
train_csv_path = str(root / "dataset/source/csv/sources_kfold_train.csv")
test_csv_path = str(root / "dataset/source/csv/sources_kfold_test.csv")
images_ref_path = str(root / "dataset/image_ref")

anonymize_dataset(train_csv_path)
download_images(train_csv_path, f"{images_ref_path}/train")

anonymize_dataset(test_csv_path)
download_images(test_csv_path, f"{images_ref_path}/test")

# Learning Curve Benchmark

## Learning Curve Classic Split

### Parameters

In [ ]:
n = 3

### Goal
Benchmark the model "yolo11-cls" model on an increasing quantity of data.

The dataset has been already tested and the model too (KFold cross validation).

Just change the training dataset quantity but keep the same test dataset
- 20% -> 40% -> 60% -> 80% -> 100

### Create datasets

In [ ]:
quantities = [0.2, 0.4, 0.6, 0.8, 1.0]
root = find_root_repository()
path = root / "dataset/benchmark"
path_csv = root / "dataset/benchmark"

for q in quantities:
    for i in range(n):
        create_dataset(str(root / "dataset/source/csv/sources_kfold_train.csv"), 
                       str(root / f"dataset/benchmark/kfold_{q}/{i}/train"), 
                       str(root / f"dataset/source/csv/benchmark/kfold_{q}/{i}.csv"), quantity=q, is_random=True)
        link_dir(str(root / "dataset/main/test"),
                 str(root / f"dataset/benchmark/kfold_{q}/{i}/test"))


### Training

In [ ]:
def benchmark_dataset_yolo(config, datasets: list[str], names, n=3):
    # mean of (score, accuracy, recall, f1)
    histories = []
    for ds_path in datasets:
        
        results = []
        if n > 1:
            for i in range(n):
                name = os.path.basename(ds_path)

                ds = Path(ds_path, str(i))

                model = train(name, ds, config)
                _, metric = test(model, ds)

                results.append(metric[0])

            results = np.asarray(results)
            histories.append([(results.mean(), results.std())])
        else:
            name = os.path.basename(ds_path)
            model = train(name, ds_path, config)
            _, metric = test(model, ds_path)
            histories.append([(metric[0], 0)])
        
    print(histories)
    plot_metrics(histories, names, metrics=[("accuracy", "number of images")])

In [ ]:
config = {"model": str(root / "models/yolo11x-cls.pt"), 
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512,
        }

root = find_root_repository()

# benchmark_dataset_yolo(config, ["dataset/benchmark/kfold_0.2/"], ["20"])
benchmark_dataset_yolo(config, [str(root / "dataset/benchmark/kfold_0.2/"), 
                                str(root / "dataset/benchmark/kfold_0.4/"), 
                                str(root / "dataset/benchmark/kfold_0.6/"), 
                                str(root / "dataset/benchmark/kfold_0.8/"), 
                                str(root / "dataset/benchmark/kfold_1.0/")], 
                       ["20", "40", "60", "80", "100"])

## Learning Curve: LogScale

#### Dataset Generation

In [ ]:
root = find_root_repository()

train_set = str(root / "dataset/source/csv/sources_kfold_train.csv")

# split
[df_train, df_val] = split_dataset(train_set, [80, 20], random=True)
output_dir = str(root / "dataset/benchmark/learning_curve")
output_csv = str(root / "dataset/source/csv/benchmark/learning_curve")

create_dataset(df_val, f"{output_dir}/val", f"{output_csv}/val.csv")

for size in L:
    create_dataset(df_train, f"{output_dir}/{size}/train", f"{output_csv}/{size}.csv", 
                   size, is_random=True, is_subject_level=True)
    
    link_dir(f"{output_dir}/val",
            f"{output_dir}/{size}/val")

### Training

In [ ]:
root = find_root_repository()

config = {"model": str(root / "models/yolo11x-cls.pt"),
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512,
        }

dataset_path = "dataset/benchmark/learning_curve"


datasets_path = [ f"{dataset_path}/{size}" for size in L]
points = [ f"{size}" for size in L]

benchmark_dataset_yolo(config, datasets_path, points, n=1)

## Learning Curve: Cross Validation + LogScale (Final Version)

The goal is to produce a learning curve based on cross validation.

Each fold have a validation set. For each fold we train N models (where N is the number of point)
with increasing datasets (4 -> 10 -> ...), we keep add data to the training (4 -> 4 old data + 6 new data -> 10 old + 5 new data -> ...).

After all we compute the fold average result of each point of the curve

In [ ]:
def learning_curve_cv(source, cv_output_dir, cv_output_csv, points):
    # generate 5 fold
    validations = split_dataset(source, np.asarray([20, 20, 20, 20, 20]), random=True)
    folds = [] # [(val, train)]

    for i, df_val in enumerate(validations):
        df_train_lst = [chunck for j, chunck in enumerate(validations) if j != i]
        df_train = join_dataframe(*df_train_lst)
        folds.append((df_val, df_train))

    folds_with_increasing_dataset = []
    # foreach fold generate X dataset [0 -> 80]
    for df_val, df_train in folds:
        assert df_train.shape[0] == 80 and df_val.shape[0] == 20
        check_no_collision(df_val, df_train)
        all_train_df = generate_increasing_datasets(df_train, points, random=True)
        folds_with_increasing_dataset.append((df_val, all_train_df))

    metrics_all_points: list[list] = [[] for _ in range(len(points))]

    for df_val, lst_df_train in folds_with_increasing_dataset:

        create_dataset(df_val, f"{cv_output_dir}/test", f"{cv_output_csv}/test.csv", source_col="LOCAL_REF")

        for pt, df_train in enumerate(lst_df_train):

            # dataset creation
            name = str(df_train.shape[0])
            ds = f"{cv_output_dir}/{name}"
            create_dataset(df_train, f"{ds}/train", f"{cv_output_csv}/{name}.csv", source_col="LOCAL_REF")
            link_dir(f"{cv_output_dir}/test", f"{ds}/test")

            # training
            model = train(name, ds, config)
            _, metric = test(model, ds)

            metrics_all_points[pt].append(metric[0])

        # deletion folder learning_curve_cv
        shutil.rmtree(cv_output_dir)

    print(metrics_all_points)
    histories_mean = [np.asarray(l).mean() for l in metrics_all_points]
    names = [f"{i}" for i in points]
    plot_metrics(histories_mean, names, metrics=[("accuracy", "number of images")], savename=str(find_root_repository() / "other/articles/learning_curve"))
    
    print(histories_mean)
    print(names)
    
    return (histories_mean, names)

In [ ]:
root = find_root_repository()

source = root / "dataset/source/csv/sources_kfold_train.csv"
benchmark_output_dir = root / "dataset/benchmark/learning_curve_cv"
benchmark_output_csv = root / "dataset/source/csv/benchmark/learning_curve_cv"
sizes = logscale(1, 40, 20)
sizes = [2, 4, 6, 8, 10, 12, 16, 22, 28, 36, 48, 62, 80]


config = {"model": str(root / "models/yolo11n-cls.pt"),
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512,
        }

In [ ]:
learning_curve_cv(str(source), str(benchmark_output_dir), str(benchmark_output_csv), sizes)

Version in the article

In [ ]:
mesures = [np.float64(0.4900000035762787), np.float64(0.5799999892711639), np.float64(0.6200000047683716), np.float64(0.5800000071525574), np.float64(0.6099999904632568), np.float64(0.6399999976158142), np.float64(0.7), np.float64(0.9199999809265137), np.float64(0.9), np.float64(0.9399999976158142), np.float64(0.9899999976158143), np.float64(0.9799999952316284), np.float64(0.9899999976158143)]
names = [2, 4, 6, 8, 10, 12, 16, 22, 28, 36, 48, 62, 80]

plot_metrics(mesures, names, metrics=[("accuracy", "Training set size (images)")], savename="../other/articles/observation/learning_curve_points", width=8, xticks="data")
plot_metrics(mesures, names, metrics=[("accuracy", "Training set size (images)")], savename="../other/articles/observation/learning_curve_no_points", xscale="square")

# Best YOLO 11 Model Benchmark

In [ ]:
root = find_root_repository()

config = {"model": str(root / "models/yolo11x-cls.pt"),
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512}


def YOLO11_benchmark():
    models = ["n", "s", "m", "l", "x"]
    accuracies = []
    
    for model in models:
        config["model"] = str(root / f"models/yolo11{model}-cls.pt")
        hist, metrics = kfold_cross_validation(5, str(root / "dataset/kfold"), config)
        
        acc: np.ndarray = np.asarray([m[0] for m in metrics])
        mean = acc.mean()
        std = acc.std()
        
        fig = plot_kfold_report(hist, metrics, ["accuracy", "precision", "rappel", "F1 score"])
        add_text_to_report(f"model: yolo11{model}-cls.pt")
        add_fig_to_report(fig)
        
        accuracies.append((mean, std))
        
    return accuracies

YOLO11_benchmark()

## Find the number for each models (info for the article):
- number of parameters
- inference time

In [ ]:
from torch import nn
from ultralytics.engine.results import Results

config = {"model": "models/yolo11x-cls.pt",
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 5,
            "imgsz": 512
}

models = ["n", "s", "m", "l", "x"]
res = []
image = cv2.imread("sample_M0.png")

for m in models:
    name_path = f"../models/yolo11{m}-cls.pt"
    config["model"] = name_path

    model = train(f"train_{name_path}", "../dataset/main", config, "benchmark_yolo_models")
    assert isinstance(model.model, nn.Module)
    params = sum(p.numel() for p in model.model.parameters())


    def predict() -> Results:
        results = model.predict(
            source=image,
            imgsz=config["imgsz"],
            batch=1,
            device=0,
            half=False,
            verbose=False
        )
        
        assert isinstance(results, list)
        result = results[0]
        assert isinstance(result, Results)
        return result

    # warmup GPU
    for _ in range(20):
        predict()

    times = []
    for _ in range(200):
        result = predict()
        times.append(result.speed["inference"])

    res.append((np.asarray(times), params))

for i, m in enumerate(models):
    times, params = res[i]
    print(f"Model yolo11{m}-cls:")
    print(f"- Parameters: {params:,}")
    print(f"- Parameters (M): {params / 1e6:.3f}")
    print(f"- Inference time (MS) {times.mean()} +- {times.std()}")

Model yolo11n-cls:
- Parameters: 1,533,666
- Parameters (M): 1.534
- Inference time (MS) 1.482578610030032 +- 0.09749447625958439

Model yolo11s-cls:
- Parameters: 5,445,570
- Parameters (M): 5.446
- Inference time (MS) 1.7114012050660676 +- 0.06007671447505875

Model yolo11m-cls:
- Parameters: 10,355,778
- Parameters (M): 10.356
- Inference time (MS) 3.210682289882243 +- 0.009586299509247432

Model yolo11l-cls:
- Parameters: 12,837,186
- Parameters (M): 12.837
- Inference time (MS) 4.391602170107944 +- 0.012168066227223938

Model yolo11x-cls:
- Parameters: 28,358,626
- Parameters (M): 28.359
- Inference time (MS) 8.099679965125688 +- 0.011721665855120593